# Bronze — AIHW Metadata Ingestion

**Notebook:** `bronze_aihw_metadata_ingestion.ipynb`  
**Layer:** Bronze  
**Source:** AIHW MyHospitals API  
**Output:** Raw metadata JSON files in `Files/bronze/metadata/`

Retrieves the AIHW reference metadata for measure categories, measures, datasets, reporting unit types and reporting units. The original API responses are saved in the Lakehouse without transformation. The final code also inspects a sample Bronze data file to review its records and suppression fields.

In [1]:
# Set the Lakehouse context

lakehouse_name = "HospitalAnalyticsLakehouse"

print("Lakehouse configured:", lakehouse_name)
print("Default Lakehouse path:", notebookutils.runtime.context.get("defaultLakehouseName"))
print("Lakehouse attachment check complete.")

# Create the Bronze metadata folder

bronze_path = "Files/bronze"
metadata_path = f"{bronze_path}/metadata"

notebookutils.fs.mkdirs(metadata_path)

print("Bronze root:", bronze_path)
print("Metadata folder:", metadata_path)
print("Folder creation command completed.")

# Configure the API connection

import requests
import json
from datetime import datetime, timezone

base_url = "https://myhospitalsapi.aihw.gov.au/api/v1"

headers = {
    "User-Agent": "HospitalAnalyticsPortfolio/1.0",
    "Accept": "application/json"
}

print("API base URL configured:", base_url)
print("Request headers configured:", list(headers.keys()))
print("UTC run timestamp:", datetime.now(timezone.utc).isoformat())

StatementMeta(, bdb51609-57c4-496d-a6f9-eec0ded292d5, 3, Finished, Available, Finished, False)

Lakehouse configured: HospitalAnalyticsLakehouse
Default Lakehouse path: HospitalAnalyticsLakehouse
Lakehouse attachment check complete.
Bronze root: Files/bronze
Metadata folder: Files/bronze/metadata
Folder creation command completed.
API base URL configured: https://myhospitalsapi.aihw.gov.au/api/v1
Request headers configured: ['User-Agent', 'Accept']
UTC run timestamp: 2026-11-02T02:02:40.401168+00:00


In [2]:
# Retrieve measure categories

endpoint = f"{base_url}/measure-categories"

response = requests.get(
    endpoint,
    headers=headers,
    timeout=30
)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    categories_payload = response.json()

    print("Category records:", len(categories_payload.get("result", [])))
    print("Source version:", categories_payload.get("version_information"))
else:
    raise RuntimeError(
        f"Category request failed: HTTP {response.status_code}"
    )

# Save the raw categories response

run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

categories_file = (
    f"{metadata_path}/categories_{run_timestamp}.json"
)

categories_json = json.dumps(
    categories_payload,
    ensure_ascii=False,
    indent=2
)

notebookutils.fs.put(
    categories_file,
    categories_json,
    True
)

print("Saved file:", categories_file)
print("JSON characters:", len(categories_json))

# Review the saved metadata files

saved_files = notebookutils.fs.ls(metadata_path)

for file in saved_files:
    print(file.name, file.size, "bytes")

# Retrieve measure metadata

measures_endpoint = f"{base_url}/measures"

measures_response = requests.get(
    measures_endpoint,
    headers=headers,
    timeout=30
)

print("HTTP status:", measures_response.status_code)

if measures_response.status_code == 200:
    measures_payload = measures_response.json()

    measures_records = measures_payload.get("result", [])

    print("Measure records:", len(measures_records))
    print("Source version:", measures_payload.get("version_information"))
else:
    raise RuntimeError(
        f"Measures request failed: HTTP {measures_response.status_code}"
    )

# Save the raw measures response

measures_file = (
    f"{metadata_path}/measures_{run_timestamp}.json"
)

measures_json = json.dumps(
    measures_payload,
    ensure_ascii=False,
    indent=2
)

notebookutils.fs.put(
    measures_file,
    measures_json,
    True
)

print("Saved file:", measures_file)
print("JSON characters:", len(measures_json))

StatementMeta(, bdb51609-57c4-496d-a6f9-eec0ded292d5, 4, Finished, Available, Finished, False)

HTTP status: 200
Category records: 12
Source version: {'api_version': '1.6.4.0', 'data_version': 2026081901, 'date_uploaded': '2026-08-19T00:00:00', 'requested_time_stamp': '2026-11-02T13:02:44.2127472+11:00'}
Saved file: Files/bronze/metadata/categories_20261102T020244Z.json
JSON characters: 1606
categories_20261018T023939Z.json 1606 bytes
categories_20261102T020015Z.json 1606 bytes
categories_20261102T020244Z.json 1606 bytes
datasets_20261018T023939Z.json 4096660 bytes
datasets_20261102T020015Z.json 4096660 bytes
measures_20261018T023939Z.json 24880 bytes
measures_20261102T020015Z.json 24880 bytes
reporting_unit_types_20261018T023939Z.json 838 bytes
reporting_unit_types_20261102T020015Z.json 838 bytes
reporting_units_20261018T023939Z.json 3552464 bytes
reporting_units_20261102T020015Z.json 3552463 bytes
HTTP status: 200
Measure records: 33
Source version: {'api_version': '1.6.4.0', 'data_version': 2026081901, 'date_uploaded': '2026-08-19T00:00:00', 'requested_time_stamp': '2026-11-02

In [3]:
# Retrieve dataset metadata

datasets_endpoint = f"{base_url}/datasets"

datasets_response = requests.get(
    datasets_endpoint,
    headers=headers,
    timeout=60
)

print("HTTP status:", datasets_response.status_code)

if datasets_response.status_code == 200:
    datasets_payload = datasets_response.json()
    datasets_records = datasets_payload.get("result", [])

    print("Dataset records:", len(datasets_records))
    print(
        "Source data version:",
        datasets_payload.get("meta", {}).get("data_version")
    )
else:
    raise Exception(
        f"Datasets request failed: "
        f"{datasets_response.status_code} - "
        f"{datasets_response.text[:500]}"
    )

# Save the raw dataset response

datasets_file = f"{metadata_path}/datasets_{run_timestamp}.json"

datasets_json = json.dumps(
    datasets_payload,
    ensure_ascii=False,
    indent=2
)

notebookutils.fs.put(
    datasets_file,
    datasets_json,
    True
)

print("Saved file:", datasets_file)
print("JSON characters:", len(datasets_json))

StatementMeta(, bdb51609-57c4-496d-a6f9-eec0ded292d5, 5, Finished, Available, Finished, False)

HTTP status: 200
Dataset records: 7244
Source data version: None
Saved file: Files/bronze/metadata/datasets_20261102T020244Z.json
JSON characters: 4096661


In [4]:
# Retrieve and save reporting unit types

unit_types_endpoint = f"{base_url}/reporting-unit-types"

unit_types_response = requests.get(
    unit_types_endpoint,
    headers=headers,
    timeout=60
)

print("HTTP status:", unit_types_response.status_code)

if unit_types_response.status_code != 200:
    raise Exception(
        f"Request failed: {unit_types_response.status_code} - "
        f"{unit_types_response.text[:500]}"
    )

unit_types_payload = unit_types_response.json()
unit_types_records = unit_types_payload.get("result", [])

unit_types_file = f"{metadata_path}/reporting_unit_types_{run_timestamp}.json"

unit_types_json = json.dumps(
    unit_types_payload,
    ensure_ascii=False,
    indent=2
)

notebookutils.fs.put(
    unit_types_file,
    unit_types_json,
    True
)

print("Reporting unit types:", len(unit_types_records))
print("Saved file:", unit_types_file)
print("JSON characters:", len(unit_types_json))

# Retrieve and save reporting units

reporting_units_endpoint = f"{base_url}/reporting-units"

reporting_units_response = requests.get(
    reporting_units_endpoint,
    headers=headers,
    timeout=60
)

print("HTTP status:", reporting_units_response.status_code)

if reporting_units_response.status_code != 200:
    raise Exception(
        f"Request failed: {reporting_units_response.status_code} - "
        f"{reporting_units_response.text[:500]}"
    )

reporting_units_payload = reporting_units_response.json()
reporting_units_records = reporting_units_payload.get("result", [])

reporting_units_file = (
    f"{metadata_path}/reporting_units_{run_timestamp}.json"
)

reporting_units_json = json.dumps(
    reporting_units_payload,
    ensure_ascii=False,
    indent=2
)

notebookutils.fs.put(
    reporting_units_file,
    reporting_units_json,
    True
)

print("Reporting units:", len(reporting_units_records))
print("Saved file:", reporting_units_file)
print("JSON characters:", len(reporting_units_json))

StatementMeta(, bdb51609-57c4-496d-a6f9-eec0ded292d5, 6, Finished, Available, Finished, False)

HTTP status: 200
Reporting unit types: 6
Saved file: Files/bronze/metadata/reporting_unit_types_20261102T020244Z.json
JSON characters: 837
HTTP status: 200
Reporting units: 1428
Saved file: Files/bronze/metadata/reporting_units_20261102T020244Z.json
JSON characters: 3552464


In [6]:
# Verify the saved metadata files

metadata_files = notebookutils.fs.ls(metadata_path)

for file in metadata_files:
    print(f"{file.name} | {file.size:,} bytes")

print("\nTotal files:", len(metadata_files))

expected_files = [
    "categories_",
    "measures_",
    "datasets_",
    "reporting_unit_types_",
    "reporting_units_"
]

for prefix in expected_files:
    found = any(
        file.name.startswith(prefix) and file.name.endswith(".json")
        for file in metadata_files
    )
    print(f"{prefix}: {'FOUND' if found else 'MISSING'}")

if not all(
    any(
        file.name.startswith(prefix) and file.name.endswith(".json")
        for file in metadata_files
    )
    for prefix in expected_files
):
    raise Exception("One or more expected metadata files are missing.")

print("\nMetadata verification completed successfully.")

StatementMeta(, bdb51609-57c4-496d-a6f9-eec0ded292d5, 8, Finished, Available, Finished, False)

categories_20261018T023939Z.json | 1,606 bytes
categories_20261102T020015Z.json | 1,606 bytes
categories_20261102T020244Z.json | 1,606 bytes
datasets_20261018T023939Z.json | 4,096,660 bytes
datasets_20261102T020015Z.json | 4,096,660 bytes
datasets_20261102T020244Z.json | 4,096,661 bytes
measures_20261018T023939Z.json | 24,880 bytes
measures_20261102T020015Z.json | 24,880 bytes
measures_20261102T020244Z.json | 24,880 bytes
reporting_unit_types_20261018T023939Z.json | 838 bytes
reporting_unit_types_20261102T020015Z.json | 838 bytes
reporting_unit_types_20261102T020244Z.json | 837 bytes
reporting_units_20261018T023939Z.json | 3,552,464 bytes
reporting_units_20261102T020015Z.json | 3,552,463 bytes
reporting_units_20261102T020244Z.json | 3,552,464 bytes

Total files: 15
categories_: FOUND
measures_: FOUND
datasets_: FOUND
reporting_unit_types_: FOUND
reporting_units_: FOUND

Metadata verification completed successfully.
